# Netflix Data Analysis

## Objectives
- Handle missing values, outliers, and duplicates
- Use Pandas, Matplotlib, and Seaborn for analysis
- Create dashboards or visual reports of key findings
- Learn data preprocessing, visualization, and storytelling with data

## 1. Load and Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
df = pd.read_csv('../data/raw/netflix_titles.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 2. Handle Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna('Unknown', inplace=True)
df['rating'].fillna(df['rating'].mode()[0], inplace=True)
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

print('Missing values after cleaning:')
print(df.isnull().sum())

## 3. Handle Duplicates

In [ ]:
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

In [ ]:
df.drop_duplicates(inplace=True)
print(f'Dataset shape after removing duplicates: {df.shape}')

## 4. Handle Outliers

In [ ]:
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')

plt.figure(figsize=(10, 6))
sns.boxplot(x=df['release_year'])
plt.title('Boxplot of Release Year')
plt.show()

In [ ]:
Q1 = df['release_year'].quantile(0.25)
Q3 = df['release_year'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['release_year'] < lower_bound) | (df['release_year'] > upper_bound)]
print(f'Outliers detected: {len(outliers)}')
print(f'Bounds: {lower_bound:.0f} - {upper_bound:.0f}')

In [ ]:
df_clean = df[(df['release_year'] >= lower_bound) & (df['release_year'] <= upper_bound)]
print(f'Dataset shape after removing outliers: {df_clean.shape}')

## 5. Feature Engineering

In [ ]:
df_clean['year_added'] = df_clean['date_added'].dt.year
df_clean['month_added'] = df_clean['date_added'].dt.month
df_clean['primary_country'] = df_clean['country'].apply(lambda x: x.split(',')[0].strip())
df_clean['primary_genre'] = df_clean['listed_in'].apply(lambda x: x.split(',')[0].strip())

df_clean.head()

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

type_counts = df_clean['type'].value_counts()
axes[0].bar(type_counts.index, type_counts.values, color=['#E50914', '#221F1F'])
axes[0].set_title('Content Type Distribution')
axes[0].set_ylabel('Count')

axes[1].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
            colors=['#E50914', '#221F1F'], startangle=90)
axes[1].set_title('Content Type Pie Chart')

plt.tight_layout()
plt.show()

In [ ]:
yearly = df_clean['release_year'].value_counts().sort_index()

plt.figure(figsize=(14, 6))
plt.plot(yearly.index, yearly.values, marker='o', color='#E50914', linewidth=2)
plt.fill_between(yearly.index, yearly.values, alpha=0.3, color='#E50914')
plt.title('Number of Releases by Year', fontsize=16)
plt.xlabel('Year')
plt.ylabel('Number of Titles')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
countries = df_clean['primary_country'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=countries.values, y=countries.index, palette='Reds_r')
plt.title('Top 10 Countries by Content', fontsize=16)
plt.xlabel('Number of Titles')
plt.tight_layout()
plt.show()

In [ ]:
genres = df_clean['primary_genre'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=genres.values, y=genres.index, palette='Reds_r')
plt.title('Top 10 Genres', fontsize=16)
plt.xlabel('Number of Titles')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ratings = df_clean['rating'].value_counts().head(10)
axes[0].barh(ratings.index, ratings.values, color='#E50914')
axes[0].set_title('Top 10 Ratings')
axes[0].set_xlabel('Count')

monthly = df_clean['month_added'].value_counts().sort_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1].bar(month_names, monthly.values, color='#E50914')
axes[1].set_title('Monthly Additions')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
directors = df_clean['director'].str.split(',').explode().str.strip()
directors = directors[directors != 'Unknown'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=directors.values, y=directors.index, palette='Reds_r')
plt.title('Top 10 Directors', fontsize=16)
plt.xlabel('Number of Titles')
plt.tight_layout()
plt.show()

In [ ]:
actors = df_clean['cast'].str.split(',').explode().str.strip()
actors = actors[actors != 'Unknown'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=actors.values, y=actors.index, palette='Reds_r')
plt.title('Top 10 Actors', fontsize=16)
plt.xlabel('Number of Titles')
plt.tight_layout()
plt.show()

## 7. Dashboard: Combined View

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Netflix Content Dashboard', fontsize=20, fontweight='bold', y=1.02)

ax1 = fig.add_subplot(3, 3, 1)
df_clean['type'].value_counts().plot(kind='bar', ax=ax1, color=['#E50914', '#221F1F'])
ax1.set_title('Content Type')
ax1.set_ylabel('Count')

ax2 = fig.add_subplot(3, 3, 2)
yearly.plot(kind='line', ax=ax2, color='#E50914', marker='o')
ax2.set_title('Releases by Year')
ax2.set_ylabel('Count')

ax3 = fig.add_subplot(3, 3, 3)
countries.head(5).plot(kind='barh', ax=ax3, color='#E50914')
ax3.set_title('Top 5 Countries')

ax4 = fig.add_subplot(3, 3, 4)
genres.head(5).plot(kind='barh', ax=ax4, color='#E50914')
ax4.set_title('Top 5 Genres')

ax5 = fig.add_subplot(3, 3, 5)
ratings.head(5).plot(kind='barh', ax=ax5, color='#E50914')
ax5.set_title('Top 5 Ratings')

ax6 = fig.add_subplot(3, 3, 6)
monthly.plot(kind='bar', ax=ax6, color='#E50914')
ax6.set_title('Monthly Additions')
ax6.set_xticklabels(month_names, rotation=45)

plt.tight_layout()
plt.savefig('../output/charts/dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Findings

In [ ]:
print('=' * 60)
print('KEY FINDINGS')
print('=' * 60)
print(f'\n1. Total Titles: {len(df_clean)}')
print(f'2. Movies: {len(df_clean[df_clean["type"]=="Movie"])} ({len(df_clean[df_clean["type"]=="Movie"])/len(df_clean)*100:.1f}%)')
print(f'3. TV Shows: {len(df_clean[df_clean["type"]=="TV Show"])} ({len(df_clean[df_clean["type"]=="TV Show"])/len(df_clean)*100:.1f}%)')
print(f'4. Top Country: {countries.index[0]} ({countries.values[0]} titles)')
print(f'5. Top Genre: {genres.index[0]} ({genres.values[0]} titles)')
print(f'6. Peak Year: {yearly.idxmax()} ({yearly.max()} releases)')
print(f'7. Most Common Rating: {ratings.index[0]}')
print(f'8. Data Range: {df_clean["release_year"].min():.0f} - {df_clean["release_year"].max():.0f}')
print('=' * 60)

In [ ]:
df_clean.to_csv('../data/processed/netflix_titles_cleaned.csv', index=False)
print('Cleaned data saved to data/processed/netflix_titles_cleaned.csv')